In [3]:
# Imports
import numpy as np
import time
from datetime import datetime
import os

from mcg_lab_vna_control import connect_vna

# Conexión transparente al VNA Keysight
vna = connect_vna(brand='keysight')

def time_sleep_sweep_time(n_sweeps_multiplier):
    sweep_time = vna.query_sweep_time(verbose=False)
    time.sleep(n_sweeps_multiplier * sweep_time + 1)

def time_now():
    return f"[{datetime.now().strftime('%H:%M:%S')}]"

def get_timestamp():
    return f"{datetime.now().strftime('%H:%M:%S')}"

ModuleNotFoundError: No module named 'numpy'

In [ ]:
########## Parameters ##########

# Carpeta para guardar las medidas manuales
folder_name = 'data/VA_manual_15_to_30GHz'

# VNA Parameters (15 a 30 GHz)
f_start = 15.0e9       # 15 GHz
f_stop = 30.0e9        # 30 GHz
points = 3001          # 3001 puntos (resolución de 5 MHz)
ifbw = 10e3            # 10 kHz
pow_dbm_list = [-15, 0, 0, 0] # Potencia en puertos [1, 2, 3, 4] en dBm
n_sweeps = 10          # 10 barridos por posición
n_average = 1

# Parámetros S:
# - Biestáticos (Tx Puerto 1 -> Rx Puertos 2, 3 y 4): S21, S31, S41
# - Monoestáticos (Reflexión / Contacto de las 4 antenas): S11, S22, S33, S44
sparam_list = ['S21', 'S31', 'S41', 'S11', 'S22', 'S33', 'S44']
sparam_list_bistatic = sparam_list[0:3]   # ['S21', 'S31', 'S41'] -> Se miden de forma rápida
extra_sparam_list = sparam_list[3:]       # ['S11', 'S22', 'S33', 'S44'] -> Reflexiones de control
extra_display_sparam_list = [False] * len(extra_sparam_list)

n_sparams = len(sparam_list)
trace_list_all = [v + 1 for v in range(n_sparams)] # [1, 2, 3, 4, 5, 6, 7]
trace_list_bistatic = [1, 2, 3]                    # [1, 2, 3]

# Definición de la línea o puntos manuales a medir (ej. 10 posiciones horizontales)
setup_number = 1
va_Nh = 10 
va_nv_index = 0 # Índice de la fila vertical actual

################----------------#

if not os.path.exists(folder_name):
    os.makedirs(folder_name, exist_ok=True)

conf_params = {
    'ifbw': ifbw,
    'pow_dbm_list': pow_dbm_list,
    'n_sweeps': n_sweeps,
    'n_average': n_average,
    'sparam_list': sparam_list,
    'f_start': f_start,
    'f_stop': f_stop,
    'points': points
}

In [ ]:
vna.preset()

# Configuramos de base las 3 trazas biestáticas principales para optimizar la velocidad
vna.configure_multiple_sparams(
    sparam_list=sparam_list_bistatic,
    f_start=f_start,
    f_stop=f_stop,
    points=points,
    ifbw=ifbw,
    pow_dbm_list=pow_dbm_list
)
vna.calibration_on()
vna_freqs = vna.get_freq_vector()

time_sleep_sweep_time(n_sweeps_multiplier=3)
vna.auto_scale_all()
vna.run_cont()

print(f"✅ VNA configurado de 15 a 30 GHz ({len(vna_freqs)} puntos). Listo para medida manual.")

In [ ]:
# Preparamos la estructura para almacenar la fila completa de 'va_Nh' posiciones
freq_vector = vna.get_freq_vector()
freq_points = len(freq_vector)

if n_average == 1:
    vna.set_sweeps(sweeps=1)
    vna.unset_average()
    n_sweeps_loop = n_sweeps
    row_sparams = np.zeros((va_Nh, n_sweeps, len(trace_list_all), freq_points), dtype=np.complex64)
else:
    vna.set_sweeps(sweeps=n_sweeps)
    vna.set_average(avg_sweeps=n_average)
    n_sweeps_loop = 1
    row_sparams = np.zeros((va_Nh, 1, len(trace_list_all), freq_points), dtype=np.complex64)

timestamp_list = [["" for _ in range(va_Nh)] for _ in range(n_sweeps_loop)]
print(f"Estructura inicializada para {va_Nh} posiciones manuales.")

In [ ]:
############### ACTUALIZA ESTE VALOR EN CADA POSICIÓN MANUAL ###############
step_i = 0  # Cambiar de 0 a 9 a medida que avanzas en las 10 posiciones
###########################################################################

print(f"{time_now()} Midiendo posición manual index = {step_i}...")

# En los extremos (ej. inicio y fin de la línea) medimos también las reflexiones (S11, S22, S33, S44)
measure_reflections = (step_i == 0 or step_i == va_Nh - 1)

if measure_reflections:
    vna.add_multiple_sparams(sparam_list=extra_sparam_list, display_sparam_list=extra_display_sparam_list)

for sweep in range(n_sweeps_loop):
    if measure_reflections:
        sparams = vna.measure_multiple_traces_sparams(trace_list=trace_list_all)
        row_sparams[step_i, sweep, :, :] = sparams
    else:
        # Medida rápida: solo transmisión S21, S31, S41
        sparams = vna.measure_multiple_traces_sparams(trace_list=trace_list_bistatic)
        row_sparams[step_i, sweep, 0:3, :] = sparams

    timestamp_list[sweep][step_i] = get_timestamp()

if measure_reflections:
    vna.delete_multiple_sparams(trace_list=trace_list_all[3:])

# Si es la última posición de la fila, guarda automáticamente el archivo .npz
if step_i == va_Nh - 1:
    filename_row = f"{folder_name}/setup_{setup_number}_nv_{va_nv_index}"
    np.savez(
        filename_row,
        row_sparams=row_sparams,
        conf_params=conf_params,
        timestamp_list=timestamp_list
    )
    print(f"💾 Fila completa guardada en: {filename_row}.npz")

print(f"✅ Posición manual i = {step_i} finalizada con éxito.")